# einops-repeat-broadcast — ex1: every-ray-with-every-triangle pairing without copy

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `einops-repeat-broadcast`. Running the final beacon cell reports progress against the `Einops: Repeat-as-broadcast` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Einops: Repeat-as-broadcast` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`einops-repeat-broadcast`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "einops-repeat-broadcast"
DD_SUBTOPIC = "Einops: Repeat-as-broadcast"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Einops repeat as broadcast — quick refresher

`einops.repeat(x, '... -> ... new', new=N)` is **not** a memcpy when used for the pure broadcast pattern. Under the hood, einops compiles it to a `torch.Tensor.expand` (or numpy stride trick) that sets the **stride of the new axis to zero** — every position along the new axis points at the same underlying storage cell.

**Three names for the same trick:**
- `einops.repeat(x, 'b d -> b n d', n=N)`
- `x.unsqueeze(1).expand(-1, N, -1)`
- `x[:, None, :].broadcast_to((x.shape[0], N, x.shape[1]))`

All three produce a view with `stride=0` on the inserted axis. No memory is allocated for the duplicates — they're a single value read N times.

**When this is the right call.** Anywhere you need to *pair every X with every Y* (e.g. every ray with every triangle in ARENA's ray tracer), reach for `einops.repeat` — it produces the expanded view in O(1) memory.

**Compared to `repeat` that actually copies.** `einops.repeat(x, 'b d -> b (n d)', n=N)` *does* materialise the copy because the output shape collapses the repeat axis into another. Only patterns that **insert** a new axis (and leave it un-grouped) stay at stride 0.

### Exercise 1 — every-ray-with-every-triangle pairing without copy

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply `einops.repeat` to broadcast a `(NT, 3, 3)` triangle batch against an `(NR, 2, 3)` ray batch so every (ray, triangle) pair becomes one slot of an `(NR, NT, ...)` tensor — without materialising the copies.
> Keywords: broadcast, pair-every-with-every, stride-zero, ray-tracing
> ```

**KCs targeted:** `repeat-inserts-zero-stride-axis`, `repeat-pair-every-with-every`

Implement `ex1_pair_rays_with_triangles(rays, triangles)`.

- `rays` has shape `(NR, 2, 3)` — `NR` rays, each `(origin, direction)` of length 3.
- `triangles` has shape `(NT, 3, 3)` — `NT` triangles, each `(A, B, C)` of length 3.

Return a tuple `(rays_b, tris_b)` where:
- `rays_b.shape == (NR, NT, 2, 3)` — every ray paired against every triangle.
- `tris_b.shape == (NR, NT, 3, 3)` — every triangle paired against every ray.

**Constraint — no copies.** Use `einops.repeat` to insert the new axis. After your call, the returned tensors must share storage with the original inputs (the test asserts `data_ptr()` equality). That's only true if the new axis has stride 0; using `.repeat()` (the torch method, which copies) or any pattern that *groups* the new axis (`(NR NT)`) will fail the storage check.

**Patterns to use:**
- `einops.repeat(rays, 'nr p d -> nr nt p d', nt=NT)`
- `einops.repeat(triangles, 'nt p d -> nr nt p d', nr=NR)`

In [ ]:
def ex1_pair_rays_with_triangles(
    rays: Tensor, triangles: Tensor
) -> tuple[Tensor, Tensor]:
    """Broadcast every ray with every triangle. No copy."""
    raise NotImplementedError()


def _test_ex1():
    NR, NT = 4, 5
    rays      = t.randn(NR, 2, 3)
    triangles = t.randn(NT, 3, 3)

    rays_b, tris_b = ex1_pair_rays_with_triangles(rays, triangles)
    assert rays_b.shape == (NR, NT, 2, 3), f'rays_b: {tuple(rays_b.shape)}'
    assert tris_b.shape == (NR, NT, 3, 3), f'tris_b: {tuple(tris_b.shape)}'

    # --- No copy check ---
    assert rays_b.data_ptr() == rays.data_ptr(), (
        'rays_b must be a stride-0 view of rays (no memory copy). '
        'Did you accidentally call .repeat()/.contiguous()/.clone()?'
    )
    assert tris_b.data_ptr() == triangles.data_ptr(), (
        'tris_b must be a stride-0 view of triangles'
    )

    # --- Value check: every slice along the broadcast axis is identical ---
    for r in range(NR):
        for tri in range(NT):
            assert t.equal(rays_b[r, tri], rays[r]), f'rays mis-broadcast at ({r},{tri})'
            assert t.equal(tris_b[r, tri], triangles[tri]), f'tris mis-broadcast at ({r},{tri})'

    # --- Stride-0 check on the inserted axis ---
    # rays_b inserted axis is `nt` (position 1); tris_b inserted axis is `nr` (position 0).
    assert rays_b.stride()[1] == 0, f'rays_b axis-1 stride must be 0, got {rays_b.stride()[1]}'
    assert tris_b.stride()[0] == 0, f'tris_b axis-0 stride must be 0, got {tris_b.stride()[0]}'

    # --- Smoke test at realistic ARENA scale (no allocation blowup) ---
    big_rays = t.randn(2000, 2, 3)
    big_tris = t.randn(100, 3, 3)
    br, bt = ex1_pair_rays_with_triangles(big_rays, big_tris)
    assert br.shape == (2000, 100, 2, 3) and bt.shape == (2000, 100, 3, 3)
    assert br.data_ptr() == big_rays.data_ptr() and bt.data_ptr() == big_tris.data_ptr()
    print('paired 2000 x 100 = 200,000 (ray, tri) slots with zero copies')
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_pair_rays_with_triangles(rays: Tensor, triangles: Tensor) -> tuple[Tensor, Tensor]:
    NR = rays.shape[0]
    NT = triangles.shape[0]
    rays_b = einops.repeat(rays, 'nr p d -> nr nt p d', nt=NT)
    tris_b = einops.repeat(triangles, 'nt p d -> nr nt p d', nr=NR)
    return rays_b, tris_b
```

**Storage check is the whole point.** You can produce the right shape with `rays.repeat(1, NT, 1, 1)` (torch's `.repeat()`) or with `.expand().contiguous()` — but both **materialise** the 200,000-row tensor, which at scale (ARENA's mesh exercise hits millions of pairs) explodes memory. The `data_ptr()` assertion catches both mistakes immediately.

**Why einops can do this.** `'nr p d -> nr nt p d'` declares that `nt` is a NEW axis (not present on input), and asks for size `nt=NT`. Internally einops compiles this to `x.unsqueeze(1).expand(-1, NT, -1, -1)` — `expand` is the stride-0 trick.

**Contrast with the materialising case.** `einops.repeat(rays, 'nr p d -> (nr k) p d', k=NT)` would also produce `(NR*NT, 2, 3)`-ish output, but because the repeat axis is *grouped* with `nr`, the output can't be a view — einops *does* memcpy. Rule of thumb: a fresh, ungrouped repeat axis is free; any grouping forces a copy.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()